# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules (these will be checked in the model answer review next week)
1. Every bar chart **must have a zero baseline** — no exceptions (SWD p.51)
2. Every chart **must have an insight title**, not a topic title (SWD p.29)
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names (SWD p.57)

---


## Setup — Run this cell first


In [ ]:
import os
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

OUTPUT_DIR = os.path.join(os.getcwd(), 'lecture02_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())
print(f"
Outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------
import plotly.express as px
import plotly.graph_objects as go

# Step 1: Compute average happiness score by region, sorted descending
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score', ascending=False))

print(region_avg)

# Step 2: Build horizontal bar chart
fig1 = px.bar(region_avg,
              x='Happiness_Score', y='Region',
              orientation='h',
              color='Happiness_Score',
              color_continuous_scale='Blues',
              labels={'Happiness_Score': 'Avg happiness score', 'Region': ''})

fig1.update_xaxes(range=[0, region_avg['Happiness_Score'].max() * 1.05])

top_region = region_avg.iloc[0]['Region']
fig1.update_layout(
    template='simple_white',
    title_text=f"{top_region} has the highest average happiness — regional differences matter for policy",
    title_x=0.02,
    height=420
)
fig1.update_traces(texttemplate='%{x:.2f}', textposition='inside', marker_line_width=0)

fig1.write_image(os.path.join(OUTPUT_DIR, 'task1_regional_comparison.png'))
print("Saved task1_regional_comparison.png")
fig1.show()


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build chart
country_order = combined['Country'].tolist()
color_map = {'Top 8': '#1f77b4', 'Bottom 8': '#ff7f0e'}

fig2 = px.bar(combined, x='Happiness_Score', y='Country', orientation='h',
              color='Group', color_discrete_map=color_map,
              category_orders={'Country': country_order},
              labels={'Happiness_Score': 'Happiness score', 'Country': ''})

fig2.update_xaxes(range=[0, combined['Happiness_Score'].max() * 1.05])

gap_start = bottom8['Happiness_Score'].max()
gap_end = top8['Happiness_Score'].min()
fig2.add_shape(type='rect', x0=gap_start, x1=gap_end, y0=-0.5,
               y1=len(combined) - 0.5, yref='y', xref='x',
               fillcolor='LightSalmon', opacity=0.08, layer='below', line_width=0)
fig2.add_vline(x=global_avg, line_dash='dash', line_color='gray')
fig2.add_annotation(x=global_avg, y=0, xref='x', yref='paper', showarrow=False,
                    text=f'Global avg: {global_avg:.2f}', xanchor='left', yanchor='bottom')

fig2.update_layout(
    template='simple_white',
    title_text='Top 8 vs Bottom 8: Large gap in happiness highlights global inequality',
    title_x=0.02,
    height=700
)
fig2.update_traces(marker_line_width=0)

fig2.write_image(os.path.join(OUTPUT_DIR, 'task2_top_bottom_contrast.png'))
print("Saved task2_top_bottom_contrast.png")
fig2.show()


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [ ]:
# Stretch goal — grouped bar chart
# ----------------------------------

regions = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']
sub = df[df['Region'].isin(regions)].copy()
agg = sub.groupby('Region')[['GDP', 'Freedom']].mean().reset_index()

present = [r for r in regions if r in agg['Region'].values]
agg = agg[agg['Region'].isin(present)].set_index('Region').reindex(present).reset_index()

fig3 = px.bar(agg, x='Region', y=['GDP', 'Freedom'], barmode='group',
              color_discrete_map={'GDP': '#2196F3', 'Freedom': '#FF9800'},
              labels={'value': 'Average score', 'Region': '', 'variable': 'Factor'})

fig3.update_layout(
    template='simple_white',
    title_text='Western Europe leads in both GDP and Freedom — the two pillars of happiness',
    title_x=0.02,
    height=480
)

fig3.write_image(os.path.join(OUTPUT_DIR, 'task3_grouped_subfactors.png'))
print("Saved task3_grouped_subfactors.png")
fig3.show()
